<div dir="rtl" style="text-align: right">

‏# 07 — پروژه ML تولیدی end-to-end

</div>

In [ ]:
# Persian text rendering for notebook markdown and plots
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Vazirmatn",
        "Vazir",
        "IRANSans",
        "Noto Sans Arabic",
        "Noto Naskh Arabic",
        "DejaVu Sans",
    ],
    "axes.unicode_minus": False,
})

try:
    from IPython.display import HTML, display

    display(HTML('<div dir="rtl" style="text-align: right"></div>'))
except Exception:
    pass

<div dir="rtl" style="text-align: right">

‏## 1. صورت‌بندی مسئله

</div>

<div dir="rtl" style="text-align: right">

‏## Learning objectives

‏By the end of this notebook, students should be able to:

‏- Frame an ML project as a business decision, not only a prediction task.
‏- Build leakage-safe preprocessing and modeling pipelines with scikit-learn.
‏- Establish simple baselines before comparing stronger models.
‏- Evaluate classification models with metrics that match the business problem.
‏- Choose a practical decision threshold using validation data only.
‏- Interpret model behavior enough to make a responsible recommendation.
‏- Save and reload a trained pipeline for reproducible inference.

‏> **Teaching note:** Keep asking: “What decision will this model change?” If the answer is unclear, better metrics will not fix the project.

</div>

<div dir="rtl" style="text-align: right">

‏## 2. بارگذاری داده محلی

</div>

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionبازخوانیDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42

def find_project_root(start_path):
    """Find the course project root from either the repo root or notebooks folder."""
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw" / "bank-full.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/bank-full.csv from the current path")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

pd.set_option("display.max_columns", 50)

<div dir="rtl" style="text-align: right">

‏یک ممیزی سریع کمک می‌کند قبل از مدل‌سازی، توازن target، نوع ویژگی‌ها و ریسک‌های آشکار leakage را بفهمیم.

</div>

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected dataset at {DATA_PATH}")

bank = pd.read_csv(DATA_PATH, sep=";")
print(bank.shape)
bank.head()

<div dir="rtl" style="text-align: right">

‏## 3. آماده‌سازی ویژگی‌ها و split داده

</div>

In [ ]:
target_counts = bank["y"].value_counts()
target_rate = bank["y"].eq("yes").mean()

print("شمارش‌های هدف:")
print(target_counts)
print(f"\nنرخ اشتراک: {target_rate:.2%}")

bank.info()

<div dir="rtl" style="text-align: right">

‏## 4. پیش‌پردازش بدون نشت

</div>

In [ ]:
target = bank["y"].map({"no": 0, "yes": 1})
features = bank.drop(columns=["y", "duration"])

X_train, X_temp, y_train, y_temp = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape, "نرخ مثبت:", y_train.mean().round(3))
print("Validation:", X_val.shape, "نرخ مثبت:", y_val.mean().round(3))
print("Test:", X_test.shape, "نرخ مثبت:", y_test.mean().round(3))

<div dir="rtl" style="text-align: right">

‏## 5. ساخت baselineهای عملی

</div>

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_for_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

numeric_for_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocess_for_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_for_linear, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

preprocess_for_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_for_tree, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

print("ویژگی‌های عددی:", numeric_features)
print("ویژگی‌های categorical:", categorical_features)

<div dir="rtl" style="text-align: right">

‏## 6. مقایسه مدل‌ها روی داده validation

</div>

In [ ]:
models = {
    "dummy_majority": Pipeline(
        steps=[
            ("preprocess", preprocess_for_linear),
            ("model", DummyClassifier(strategy="most_frequent")),
        ]
    ),
    "logistic_regression": Pipeline(
        steps=[
            ("preprocess", preprocess_for_linear),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    ),
    "random_forest": Pipeline(
        steps=[
            ("preprocess", preprocess_for_tree),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=120,
                    min_samples_leaf=20,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"fit شد: {name}")

<div dir="rtl" style="text-align: right">

‏منحنی‌های بصری، trade-off را برای آموزش ساده‌تر می‌کنند. یک مدل بهتر معمولاً منحنی‌ای دارد که به گوشه بالا-راست نزدیک‌تر است.

</div>

In [ ]:
def evaluate_classifier(name, model, X, y, threshold=0.50):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "model": name,
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
        "precision_at_0.50": precision_score(y, predictions, zero_division=0),
        "recall_at_0.50": recall_score(y, predictions, zero_division=0),
        "positive_rate_at_0.50": predictions.mean(),
    }

validation_results = pd.DataFrame(
    [evaluate_classifier(name, model, X_val, y_val) for name, model in models.items()]
).sort_values("average_precision", ascending=False)

validation_results

<div dir="rtl" style="text-align: right">

‏## 7. انتخاب مدل و آستانه

</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, model in models.items():
    PrecisionبازخوانیDisplay.from_estimator(model, X_val, y_val, name=name, ax=axes[0])
    RocCurveDisplay.from_estimator(model, X_val, y_val, name=name, ax=axes[1])

axes[0].set_title("منحنی دقت-بازخوانی")
axes[1].set_title("منحنی ROC")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

‏## 7. Choose a model and threshold

‏For a marketing campaign, the best threshold depends on capacity and cost. A lower threshold contacts more clients and usually increases recall, but it also creates more false positives.

‏Here we use a simple validation-set cost model:

‏- False negative cost = 5: missing a potential subscriber is costly.
‏- False positive cost = 1: contacting a client who does not subscribe has a smaller cost.

‏These numbers are teaching assumptions, not universal truth. In a real project they should come from campaign economics.

</div>

In [ ]:
chosen_model_name = validation_results.iloc[0]["model"]
chosen_model = models[chosen_model_name]

print("مدل انتخاب‌شده بر اساس average precision اعتبارسنجی:", chosen_model_name)

In [ ]:
def threshold_table(y_true, probabilities, false_negative_cost=5, false_positive_cost=1):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 19):
        predictions = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
        rows.append(
            {
                "threshold": threshold,
                "precision": precision_score(y_true, predictions, zero_division=0),
                "recall": recall_score(y_true, predictions, zero_division=0),
                "contact_rate": predictions.mean(),
                "false_positives": fp,
                "false_negatives": fn,
                "cost": false_positive_cost * fp + false_negative_cost * fn,
            }
        )
    return pd.DataFrame(rows)

val_probabilities = chosen_model.predict_proba(X_val)[:, 1]
threshold_results = threshold_table(y_val, val_probabilities)
threshold_results.sort_values("cost").head(10)

<div dir="rtl" style="text-align: right">

‏## 8. ارزیابی نهایی روی test

</div>

In [ ]:
best_threshold = threshold_results.sort_values("cost").iloc[0]["threshold"]
print(f"آستانه انتخاب‌شده: {best_threshold:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="Precision")
ax.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="بازخوانی")
ax.plot(threshold_results["threshold"], threshold_results["contact_rate"], marker="o", label="Contact rate")
ax.axvline(best_threshold, color="black", linestyle="--", label="Selected threshold")
ax.set_xlabel("آستانه تصمیم")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("trade-offهای اعتبارسنجی بر حسب آستانه")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right">

‏## 9. تفسیر

</div>

In [ ]:
test_probabilities = chosen_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

print("ROC AUC آزمون:", round(roc_auc_score(y_test, test_probabilities), 3))
print("average precision آزمون:", round(average_precision_score(y_test, test_probabilities), 3))
print("\nگزارش طبقه‌بندی در آستانه انتخاب‌شده:")
print(classification_report(y_test, test_predictions, target_names=["no", "yes"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, display_labels=["no", "yes"])
plt.title(f"Test confusion matrix at threshold {best_threshold:.2f}")
plt.show()

<div dir="rtl" style="text-align: right">

‏## 10. توصیه عملی

</div>

In [ ]:
def get_feature_names(fitted_pipeline):
    preprocessor = fitted_pipeline.named_steps["preprocess"]
    return preprocessor.get_feature_names_out()

if chosen_model_name == "logistic_regression":
    feature_names = get_feature_names(chosen_model)
    coefficients = chosen_model.named_steps["model"].coef_[0]
    importance = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
    display(importance.reindex(importance.coefficient.abs().sort_values(ascending=False).index).head(15))
else:
    feature_names = get_feature_names(chosen_model)
    importances = chosen_model.named_steps["model"].feature_importances_
    importance = pd.DataFrame({"feature": feature_names, "importance": importances})
    display(importance.sort_values("importance", ascending=False).head(15))

<div dir="rtl" style="text-align: right">

‏## 11. ذخیره و بارگذاری دوباره artifact استنتاج

</div>

In [ ]:
recommendation = {
    "selected_model": chosen_model_name,
    "selected_threshold": float(best_threshold),
    "validation_average_precision": float(validation_results.iloc[0]["average_precision"]),
    "test_roc_auc": float(roc_auc_score(y_test, test_probabilities)),
    "test_average_precision": float(average_precision_score(y_test, test_probabilities)),
    "excluded_feature": "duration",
    "reason": "duration is only known after the call and would leak future information for pre-call prioritization",
}

recommendation

<div dir="rtl" style="text-align: right">

‏تأیید بارگذاری دوباره، اشتباهات رایج استقرار را زود آشکار می‌کند: ستون‌های مفقود، ناهماهنگی preprocessing، یا آستانه‌ای که درست ذخیره نشده است.

</div>

In [ ]:
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

artifact = {
    "pipeline": chosen_model,
    "threshold": float(best_threshold),
    "required_columns": X_train.columns.tolist(),
    "target_mapping": {"no": 0, "yes": 1},
}

artifact_path = MODEL_DIR / "bank_marketing_pipeline.joblib"
metadata_path = REPORT_DIR / "bank_marketing_model_metadata.json"

joblib.dump(artifact, artifact_path)
metadata_path.write_text(json.dumps(recommendation, indent=2))

print("artifact مدل ذخیره شد در:", artifact_path)
print("فراداده ذخیره شد در:", metadata_path)

<div dir="rtl" style="text-align: right">

‏## 12. بررسی‌های سبک

</div>

In [ ]:
loaded_artifact = joblib.load(artifact_path)
loaded_pipeline = loaded_artifact["pipeline"]
loaded_threshold = loaded_artifact["threshold"]
required_columns = loaded_artifact["required_columns"]

def predict_batch(raw_data):
    raw_data = raw_data.copy()
    missing_columns = sorted(set(required_columns) - set(raw_data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    probabilities = loaded_pipeline.predict_proba(raw_data[required_columns])[:, 1]
    return pd.DataFrame(
        {
            "subscription_probability": probabilities,
            "prioritize_call": probabilities >= loaded_threshold,
        },
        index=raw_data.index,
    )

sample_predictions = predict_batch(X_test.head(5))
sample_predictions

<div dir="rtl" style="text-align: right">

‏## تمرین‌ها

</div>

In [ ]:
assert "duration" not in required_columns
assert set(sample_predictions.columns) == {"subscription_probability", "prioritize_call"}
assert sample_predictions["subscription_probability"].between(0, 1).all()
assert isinstance(loaded_threshold, float)

try:
    predict_batch(X_test.drop(columns=[required_columns[0]]).head(2))
except ValueError as error:
    print("خطای مورد انتظار اعتبارسنجی:", error)

<div dir="rtl" style="text-align: right">

‏## Exercises

‏1. Change the false-negative and false-positive costs. How does the selected threshold change?
‏2. Compare the final recommendation if you choose ROC AUC instead of average precision for model selection.
‏3. Add calibration with `CalibratedClassifierCV`. Does it change the threshold decision?
‏4. Create a smaller model using fewer features. How much performance do you lose, and is the simpler model easier to explain?
‏5. Temporarily include `duration` and compare the scores. Why would that impressive result be misleading for pre-call prioritization?
‏6. Design a monitoring plan: which input features, prediction summaries, and business outcomes should be tracked after deployment?

‏> **Closing discussion:** What evidence would convince you that this model should not be deployed yet, even if its test metrics look good?

</div>

<div dir="rtl" style="text-align: right">

‏**Estimated time:** 90-120 minutes  
‏**Prerequisites:** notebooks 00-06; model evaluation, thresholding, and serialization.

‏## Common mistakes and leakage warnings

‏- Scoring the sealed test set before the final design is frozen.
‏- Changing features or thresholds after seeing final test metrics.
‏- Treating a saved artifact as complete without validation checks.

‏## Exercises

‏1. Add one more schema check to the batch validation step.
‏2. Compare the frozen threshold with a different operating point.
‏3. **Challenge:** describe how you would monitor this model after deployment.

‏## Summary

‏A production workflow needs a data contract, a frozen threshold, a single final test evaluation, and reproducible artifacts.

‏## References

‏- [scikit-learn pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline)
‏- [joblib persistence](https://joblib.readthedocs.io/en/latest/persistence.html)

</div>